# Lesson 10: Working with LLM APIs

*Worked solutions for [the lesson page](../../lessons/10-llm-apis.html). The exercise version without solutions is `../10-llm-apis.ipynb`.*

Call hosted and local models from code like a professional, covering messages and system prompts, streaming, reliable structured output, errors and retries, token costs, caching and concurrency, then build a streaming command-line chatbot.

**Runs on:** Any computer, plus an API key. Cells that call a model are marked.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q anthropic openai pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()   # reads ANTHROPIC_API_KEY from a .env file next to this notebook, if there is one
# On Google Colab, add the key under the key icon (Secrets), then uncomment the next two lines:
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"          # check your provider's current model list
print("API key found" if os.environ.get("ANTHROPIC_API_KEY") else
      "No API key set: cells marked 'Needs an API key' will fail until you add one")

## The shape of every chat API

This lesson uses the Anthropic Python SDK for the main examples, with the OpenAI-style equivalent alongside, because that second format is also what most local and open-model servers speak. The concepts transfer to any provider.

Run in a terminal:

```bash
uv add anthropic openai pydantic python-dotenv
```

> **Model names change**
>
> Providers release new models every few months. The examples use `claude-sonnet-5-5` as a capable default and `claude-haiku-5-5` as a fast, cheap one. Check your provider's model list and keep the name in one configuration constant so swapping models is a one-line change.

## First calls

**Option: Anthropic**

In [ ]:
# Needs an API key (see the setup cell)
import anthropic
from dotenv import load_dotenv

load_dotenv()                                   # ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"

resp = client.messages.create(
    model=MODEL,
    max_tokens=500,                             # upper bound on the reply length
    system="You are a concise teaching assistant for an AI engineering course.",
    messages=[{"role": "user", "content": "Explain overfitting in two sentences."}],
)
print(resp.content[0].text)
print(resp.stop_reason)                         # "end_turn", or "max_tokens" if it was cut off
print(resp.usage.input_tokens, resp.usage.output_tokens)

**Option: OpenAI-compatible**

In [ ]:
# Needs an API key (see the setup cell)
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()                                   # OPENAI_API_KEY from .env
client = OpenAI()
resp = client.chat.completions.create(
    model="YOUR_MODEL_NAME",
    max_tokens=500,
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "Explain overfitting in two sentences."},
    ],
)
print(resp.choices[0].message.content)
print(resp.choices[0].finish_reason)            # "stop" or "length"
print(resp.usage.prompt_tokens, resp.usage.completion_tokens)

### Multi-turn conversations

Because the API is stateless, a conversation is a list you keep appending to.

In [ ]:
# Needs an API key (see the setup cell)
history = []

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    resp = client.messages.create(model=MODEL, max_tokens=800, messages=history,
                                  system="You are a helpful tutor.")
    reply = resp.content[0].text
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("What is a learning rate?"))
print(chat("What happens if mine is too high?"))   # "mine" is understood because history was resent

## Streaming

A long answer can take many seconds. Streaming sends tokens as they are generated, so users see text appear immediately. The **time to first token** is what makes an app feel fast.

In [ ]:
# Needs an API key (see the setup cell)
with client.messages.stream(
    model=MODEL, max_tokens=800,
    messages=[{"role": "user", "content": "Explain gradient descent to a new engineer."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final = stream.get_final_message()               # full message with usage after streaming
print("\n", final.usage.output_tokens, "tokens")

## Structured output you can trust

### Define the shape with Pydantic

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Ticket(BaseModel):
    category: Literal["hardware", "software", "data", "other"]
    severity: int = Field(ge=1, le=5, description="1 = cosmetic, 5 = instrument down")
    instrument_id: str | None = Field(default=None, description="e.g. 'P-117' if mentioned")
    summary: str = Field(max_length=200)

### Force the model to fill it in

A reliable pattern with the Anthropic API is to declare a "tool" whose input schema is your data shape and require the model to call it. The model's output is then the tool's arguments: JSON that matches the schema.

In [ ]:
# Needs an API key (see the setup cell)
import json

def extract_ticket(email_text: str) -> Ticket:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=500,
        tools=[{
            "name": "record_ticket",
            "description": "Record a structured support ticket.",
            "input_schema": Ticket.model_json_schema(),
        }],
        tool_choice={"type": "tool", "name": "record_ticket"},     # must call this tool
        messages=[{"role": "user", "content": f"Create a ticket from this email:\n\n{email_text}"}],
    )
    block = next(b for b in resp.content if b.type == "tool_use")
    return Ticket.model_validate(block.input)                       # raises if anything is off

t = extract_ticket("Hi, unit P-117 at the coastal site stopped logging overnight. "
                   "Power LED is on but no data since 02:00. Urgent, campaign starts Monday.")
print(t)
# category='hardware' severity=4 instrument_id='P-117' summary='Unit P-117 stopped logging ...'

> **Validate, then repair**
>
> If validation fails, send the error message back to the model ("Your output failed validation: severity must be at most 5. Fix it.") and retry once or twice. Libraries such as Instructor automate this loop.

## Errors, retries and rate limits

The official SDKs already retry transient errors with exponential backoff. Configure it, and handle what remains:

In [ ]:
# Needs an API key (see the setup cell)
client = anthropic.Anthropic(max_retries=4, timeout=60.0)

try:
    resp = client.messages.create(model=MODEL, max_tokens=300,
                                  messages=[{"role": "user", "content": "Hello"}])
except anthropic.RateLimitError:
    ...   # back off further, queue the work, or route to another model
except anthropic.APIConnectionError:
    ...   # network trouble: alert if persistent
except anthropic.APIStatusError as e:
    print(e.status_code, e.message)   # bad request, auth, etc.: log and fix, do not retry blindly

## Tokens, cost and speed

### Estimate before you build

Providers price input and output tokens separately, and output tokens cost several times more than input tokens. A back-of-envelope calculation before you build saves surprises:

In [ ]:
def monthly_cost(requests_per_day, in_tokens, out_tokens, price_in_per_m, price_out_per_m):
    per_request = in_tokens / 1e6 * price_in_per_m + out_tokens / 1e6 * price_out_per_m
    return per_request * requests_per_day * 30

# 5,000 requests a day, 3,000 tokens of prompt (instructions + retrieved context), 400 tokens out.
# Plug in your provider's current per-million-token prices:
print(monthly_cost(5_000, 3_000, 400, price_in_per_m=3.0, price_out_per_m=15.0))   # 2250.0 per month

Count tokens exactly with the provider's tokenizer or counting endpoint rather than guessing:

In [ ]:
# Needs an API key (see the setup cell)
long_text = "Overfitting happens when a model memorises noise in its training data. " * 200
count = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": long_text}])
print(count.input_tokens)

### Prompt caching

If many requests share a long prefix (a big system prompt, a manual, a set of examples), providers can cache the processed prefix. Cached input tokens are billed at a fraction of the normal price and processed faster. With the Anthropic API you mark where the cacheable prefix ends:

In [ ]:
import os
if not os.path.exists("instrument_manual.txt"):   # a stand-in manual so the cell runs; use a real one
    section = ("Section 4.2, replacing the filter wheel belt: power down, remove the cover, fit belt FW-BELT-02 "
               "and tighten the motor mount screws to 0.8 N·m. Section 5.1, desiccant: replace every six months, "
               "or every three months at humid sites. ")
    open("instrument_manual.txt", "w").write(section * 150)   # several thousand tokens, enough to cache
manual = open("instrument_manual.txt").read()          # tens of thousands of tokens, same every call

def ask_manual(question: str) -> str:
    resp = client.messages.create(
        model=MODEL, max_tokens=600,
        system=[
            {"type": "text", "text": "Answer using only the manual below. Quote section numbers."},
            {"type": "text", "text": manual, "cache_control": {"type": "ephemeral"}},
        ],
        messages=[{"role": "user", "content": question}],
    )
    u = resp.usage
    print("cache write", u.cache_creation_input_tokens, "cache read", u.cache_read_input_tokens)
    return resp.content[0].text

In [ ]:
# Needs an API key (see the setup cell)
print(ask_manual("How tight should the motor mount screws be?"))   # first call writes the cache
print(ask_manual("How often is the desiccant replaced at humid sites?"))   # second call reads it

### Concurrency and batch processing

To process thousands of documents, run requests concurrently, with a limit so you stay under your rate limits.

In [ ]:
import asyncio
from anthropic import AsyncAnthropic

aclient = AsyncAnthropic(max_retries=5)
sem = asyncio.Semaphore(8)                        # at most 8 requests in flight

async def summarise(doc: str) -> str:
    async with sem:
        r = await aclient.messages.create(model="claude-haiku-5-5", max_tokens=200,
                                          messages=[{"role": "user", "content": f"Summarise in one sentence:\n{doc}"}])
        return r.content[0].text

async def main(docs):
    return await asyncio.gather(*(summarise(d) for d in docs))

# summaries = asyncio.run(main(docs))

In [ ]:
# Needs an API key (see the setup cell)
docs = ["The instrument stopped logging overnight after a failed upload and needed a reboot.",
        "Replace the desiccant every six months, or every three months at humid sites."]
summaries = await main(docs)     # in a notebook, use await instead of asyncio.run
print(summaries)

## Local and open models

Many local servers (Ollama, vLLM, llama.cpp's server, LM Studio) expose an **OpenAI-compatible** endpoint, so the same client code works by changing the base URL.

Run in a terminal:

```bash
# install Ollama from ollama.com, then:
ollama pull llama3.2          # a small open model, a couple of GB
```

In [ ]:
# Needs a running server (see the text above)
from openai import OpenAI

local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # the key is ignored locally
r = local.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": "Give me three uses of a spectrometer."}],
)
print(r.choices[0].message.content)

### Hands-on: a streaming CLI chatbot

Build `chat.py`, a terminal chatbot that:

1. Streams replies as they are generated.
2. Keeps conversation history, and trims the oldest turns when the history exceeds about 20,000 tokens (estimate with `count_tokens`, or roughly 4 characters per token).
3. Prints the running token usage and an estimated cost after each reply.
4. Supports `/reset` to clear history, `/system <text>` to change the system prompt, and `/quit`.
5. Survives a network error without losing the conversation.

#### Solution: a solution

In [ ]:
%%writefile chat.py
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(max_retries=3)
MODEL = "claude-sonnet-5-5"
PRICE_IN, PRICE_OUT = 3.0, 15.0          # per million tokens: set to your provider's current prices
MAX_HISTORY_CHARS = 80_000                # about 20k tokens

system = "You are a friendly, precise tutor for an AI engineering course."
history: list[dict] = []
spent_in = spent_out = 0

def trim(history):
    while sum(len(m["content"]) for m in history) > MAX_HISTORY_CHARS and len(history) > 2:
        history.pop(0); history.pop(0)     # drop the oldest user/assistant pair
    return history

while True:
    try:
        user = input("\nyou> ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not user:
        continue
    if user == "/quit":
        break
    if user == "/reset":
        history.clear(); print("history cleared"); continue
    if user.startswith("/system "):
        system = user[len("/system "):]; print("system prompt updated"); continue

    history.append({"role": "user", "content": user})
    trim(history)
    print("bot> ", end="", flush=True)
    try:
        with client.messages.stream(model=MODEL, max_tokens=1000, system=system,
                                    messages=history) as stream:
            parts = []
            for piece in stream.text_stream:
                print(piece, end="", flush=True)
                parts.append(piece)
            final = stream.get_final_message()
        reply = "".join(parts)
    except anthropic.APIError as e:
        history.pop()                      # remove the unanswered user turn so history stays valid
        print(f"\n[request failed: {e.__class__.__name__}; your conversation is intact, try again]")
        continue
    history.append({"role": "assistant", "content": reply})
    spent_in += final.usage.input_tokens
    spent_out += final.usage.output_tokens
    cost = spent_in / 1e6 * PRICE_IN + spent_out / 1e6 * PRICE_OUT
    print(f"\n[{final.usage.input_tokens} in / {final.usage.output_tokens} out; session ≈ {cost:.4f} USD]")

Run the chatbot in a terminal with `python chat.py`; `input()` loops do not work well inside notebooks.

Watch the input-token count climb every turn: that is the stateless API re-reading the whole history. It is the clearest demonstration of why long conversations get slow and expensive, and why trimming, summarising or caching matters.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/10-llm-apis.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.